# Force closure
- Cartesian Impedance position control + PI force control

> Flow
1. Declare xml: box with contact site (L/R)
2. Get targets: contact position, contact force direction, contact force magnitude
3. Define each controller: position with PD, contact force with PI

#### 0. Generate Scene

In [1]:
import os
import sys
import numpy as np
import time
import mujoco
sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.SPEC_HELPER import *
from pp_base_mujoco.KINEMATICS import *

In [2]:
spec_helper = MjSpecHelper()
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, 0.7, 0),
    # r=(0, 0, -1.57),
    r=(0, 0, 0),
    prefix="",
    suffix="_right"
)
spec_helper.add_robot(
    path='../asset/panda/panda_ee_sphere.xml',
    body_name="base",
    p=(0, -0.7, 0),
    # r=(0, 0, 1.57),
    r=(0, 0, 0),
    prefix="",
    suffix="_left"
)
spec_helper.add_geom(
    name="box",
    type='box',
    size=(0.15, 0.15, 0.15),
    freejoint = True,
    p=(0.2, 0, 0.15),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 0.3, 0.5),
    group=1,
    friction=(1.0, 0.05, 0.01),
    condim=4,
    solimp=(0.98, 0.999, 0.001, 0.5, 2),
    solref=(0.01, 1.0),
    mass=0.1
)
spec_helper.add_site(
    name="contact_right",
    size=(0.05,0.05,0.05),
    p=(0, 0.15+0.05, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="contact_left",
    size=(0.05,0.05,0.05),
    p=(0, -0.15-0.05, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)
spec_helper.add_site(
    name="center",
    size=(0.05,0.05,0.05),
    p=(0, 0, 0),
    r=(0, 0, 0),
    rgba=(0.3, 0.3, 1.0, 1.0),
    group=2,
    body_name="box"
)

model, data = spec_helper.compile()
spec_helper.save_to_xml("../asset/xml/scene_panda_lr.xml")

In [3]:
model.opt.timestep = 0.001

#### 1. Initialize Scene

In [4]:
joint_names = get_joint_names(model, data)
joint_names_left = [name for name in joint_names if name is not None and "_left" in name]
joint_names_right = [name for name in joint_names if name is not None and "_right" in name]

qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5])
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

p_target_contact_left = get_p(model, data, name="contact_left", type='site')
p_target_contact_right = get_p(model, data, name="contact_right", type='site')
R_target_contact_left = [[1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]]
R_target_contact_right = [[1.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]]

In [5]:
def get_jacobian_franka_ee_bimanual():
    jacobian_p_right, jacobian_r_right = get_jacobian(model, data, 'eef_sphere_right', type='geom', joints_use=joint_names_right)
    # # reduce jacobian of 3x14 to 3x7, first half for right arm 
    # jacobian_p_right = jacobian_p_right[:, :7]
    # jacobian_r_right = jacobian_r_right[:, :7]
    jacobian_p_left, jacobian_r_left = get_jacobian(model, data, 'eef_sphere_left', type='geom', joints_use=joint_names_left)
    # # reduce jacobian of 3x14 to 3x7, second half for left arm
    # jacobian_p_left = jacobian_p_left[:, 7:]
    # jacobian_r_left = jacobian_r_left[:, 7:]

    return jacobian_p_left, jacobian_r_left, jacobian_p_right, jacobian_r_right

#### 3. Reach target EE pR with IK

In [6]:
viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_init)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_init)
mujoco.mj_forward(model, data)

# IK tracking loop
while viewer.is_alive():
    p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
    p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
    R_ee_left = get_R(model, data, name="eef_sphere_left", type='geom')
    R_ee_right = get_R(model, data, name="eef_sphere_right", type='geom')

    p_target_contact_left = get_p(model, data, name="contact_left", type='site')
    p_target_contact_right = get_p(model, data, name="contact_right", type='site')

    pos_error_left, rotvec_error_left = get_ik_error_clipped(
        p_target=p_target_contact_left,
        r_target=R_target_contact_left,
        p_current=p_ee_left,
        r_current=R_ee_left,
    )
    pos_error_right, rotvec_error_right = get_ik_error_clipped(
        p_target=p_target_contact_right,
        r_target=R_target_contact_right,
        p_current=p_ee_right,
        r_current=R_ee_right,
    )
    error_left = np.concatenate([pos_error_left, rotvec_error_left])
    error_right = np.concatenate([pos_error_right, rotvec_error_right])

    if np.linalg.norm(pos_error_left) < 0.01 and np.linalg.norm(rotvec_error_left) < 0.01:
        qpos_left = get_qpos_with_names(model, data, names=joint_names_left)
    if np.linalg.norm(rotvec_error_right) < 0.01 and np.linalg.norm(pos_error_right) < 0.01:
        qpos_right = get_qpos_with_names(model, data, names=joint_names_right)

    jac_p_left, jac_r_left, jac_p_right, jac_r_right = get_jacobian_franka_ee_bimanual()
    jac_left = np.concatenate([jac_p_left, jac_r_left], axis=0)
    jac_right = np.concatenate([jac_p_right, jac_r_right], axis=0)
    jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
    jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)

    qpos_error_left = jac_left_inverse @ error_left
    qpos_error_right = jac_right_inverse @ error_right
    q_left_updated = get_qpos_with_names(model, data, names=joint_names_left) + qpos_error_left
    q_right_updated = get_qpos_with_names(model, data, names=joint_names_right) + qpos_error_right

    apply_qpos_names(model, data, names=joint_names_left, value=q_left_updated)
    apply_qpos_names(model, data, names=joint_names_right, value=q_right_updated)
    mujoco.mj_forward(model, data)
    viewer.render()
    time.sleep(0.05)

viewer.close()
del(viewer)

In [7]:
qpos_left_saved = qpos_left 
qpos_right_saved = qpos_right

In [8]:
def get_body_contact_force_position(
        model,
        data,
        body1_name,
        body2_name,
        ):
    body1_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body1_name)
    body2_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body2_name)
    contact_forces = []
    contact_positions = []
    for i in range(data.ncon):
        contact = data.contact[i]
        body1_in_contact_id = model.geom_bodyid[contact.geom1]
        body2_in_contact_id = model.geom_bodyid[contact.geom2]
        if (body1_in_contact_id == body1_id and body2_in_contact_id == body2_id) or (body1_in_contact_id == body2_id and body2_in_contact_id == body1_id):
            force = np.zeros(6) # (6,)
            mujoco.mj_contactForce(model,data,i,force)
            contact_forces.append(force)
            contact_positions.append(contact.pos)
    return contact_forces, contact_positions

#### 5. Iterate
- Iteratively apply two torques

In [9]:
actuator_names = get_actuator_names(model, data)
actuator_names_left = [name for name in actuator_names if name is not None and "_left" in name]
actuator_names_right = [name for name in actuator_names if name is not None and "_right" in name]

In [10]:
Kp_ee = 1000.0
Kd_ee = 500.0
Kp_force = 20.0
Ki_force = 0.1
Kp_qpos = 10.0
Kd_qpos = 2.0
Ki_qpos = 0.1
force_magnitude = 10.0

viewer = MUJOCOGLVIEWER(model, data)
viewer.view_geom(group=0, show=False)
viewer.view_geom(group=1, show=True)
viewer.options[0].flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True
render_tick = 0
mujoco.mj_resetData(model, data)
apply_qpos_names(model, data, names=joint_names_left, value=qpos_left_saved)
apply_qpos_names(model, data, names=joint_names_right, value=qpos_right_saved)
mujoco.mj_forward(model, data)

floor_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "floor")
model.geom_conaffinity[floor_geom_id] = 1
model.geom_contype[floor_geom_id] = 1

f_push_left = np.zeros(3)
f_push_right = np.zeros(3)
p_ee_target_left = get_p(model, data, name="contact_left", type='site').copy() + np.array([0.1, 0, 0])
p_ee_target_right = get_p(model, data, name="contact_right", type='site').copy() + np.array([0.1, 0, 0])

start_time = time.time()
f_accumulated_left = np.zeros(3)
f_accumulated_right = np.zeros(3)

# Main hybrid control loop
while viewer.is_alive():
    glfw.poll_events()
    if glfw.get_key(viewer.windows[0], glfw.KEY_W) == glfw.PRESS:
        p_ee_target_left += np.array([0, 0, 0.001])
        p_ee_target_right += np.array([0, 0, 0.001])
    if glfw.get_key(viewer.windows[0], glfw.KEY_S) == glfw.PRESS:
        p_ee_target_left += np.array([0, 0, -0.001])
        p_ee_target_right += np.array([0, 0, -0.001])
    if glfw.get_key(viewer.windows[0], glfw.KEY_A) == glfw.PRESS:
        p_ee_target_left += np.array([0, -0.001, 0])
        p_ee_target_right += np.array([0, -0.001, 0])
    if glfw.get_key(viewer.windows[0], glfw.KEY_D) == glfw.PRESS:
        p_ee_target_left += np.array([0, 0.001, 0])
        p_ee_target_right += np.array([0, 0.001, 0])

    if time.time() - start_time > 1:
        floor_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "floor")
        model.geom_conaffinity[floor_geom_id] = 0
        model.geom_contype[floor_geom_id] = 0

    # Jacobian update and EE impedance
    jac_p_left, jac_R_left, jac_p_right, jac_R_Right = get_jacobian_franka_ee_bimanual()
    jac_left = np.concatenate([jac_p_left, jac_R_left], axis=0)
    jac_right = np.concatenate([jac_p_right, jac_R_Right], axis=0)
    jac_left_inverse = get_pseudo_inverse(jac_left, method="svd", sigma_threshold=1e-3)
    jac_right_inverse = get_pseudo_inverse(jac_right, method="svd", sigma_threshold=1e-3)

    p_ee_left = get_p(model, data, name="eef_sphere_left", type='geom')
    p_ee_right = get_p(model, data, name="eef_sphere_right", type='geom')
    p_ee_left_error = p_ee_target_left - p_ee_left
    p_ee_right_error = p_ee_target_right - p_ee_right
    v_ee_target_left = np.zeros(3)
    v_ee_target_right = np.zeros(3)
    qvel_left = get_qvel_with_names(model, data, names=joint_names_left)
    qvel_right = get_qvel_with_names(model, data, names=joint_names_right)
    v_ee_left = jac_p_left @ qvel_left
    v_ee_right = jac_p_right @ qvel_right
    v_ee_left_error = v_ee_target_left - v_ee_left
    v_ee_right_error = v_ee_target_right - v_ee_right
    f_ee_desired_left = Kp_ee * p_ee_left_error + Kd_ee * v_ee_left_error
    f_ee_desired_right = Kp_ee * p_ee_right_error + Kd_ee * v_ee_right_error
    torque_ee_left = jac_p_left.T @ f_ee_desired_left
    torque_ee_right = jac_p_right.T @ f_ee_desired_right

    # Contact force PI control
    p_center = get_p(model, data, name="center", type='site')
    direction_push_left = (p_center - p_target_contact_left) / np.linalg.norm(p_center - p_target_contact_left)
    direction_push_right = (p_center - p_target_contact_right) / np.linalg.norm(p_center - p_target_contact_right)
    f_push_target_left = force_magnitude * direction_push_left
    f_push_target_right = force_magnitude * direction_push_right
    box_mass = model.body_mass[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "box")]
    gravity = model.opt.gravity
    f_gravity_compensation = np.array([0, 0, box_mass * gravity[2]])
    f_push_target_left += f_gravity_compensation
    f_push_target_right += f_gravity_compensation
    f_push_left, p_contact_left = get_body_contact_force_position(model, data, body1_name="right_hand_left", body2_name="box")
    f_push_right, p_contact_right = get_body_contact_force_position(model, data, body1_name="right_hand_right", body2_name="box")
    if len(f_push_left) == 0:
        f_push_left = np.zeros(3)
    else:
        f_push_left = f_push_left[0][:3]
    if len(f_push_right) == 0:
        f_push_right = np.zeros(3)
    else:
        f_push_right = f_push_right[0][:3]

    f_push_error_left = f_push_target_left - f_push_left
    f_push_error_right = f_push_target_right - f_push_right
    f_accumulated_left += f_push_error_left * 0.01
    f_accumulated_right += f_push_error_right * 0.01
    f_push_desired_left = f_push_target_left + Kp_force * f_push_error_left + Ki_force * f_accumulated_left
    f_push_desired_right = f_push_target_right + Kp_force * f_push_error_right + Ki_force * f_accumulated_right
    torque_push_left = jac_p_left.T @ f_push_desired_left
    torque_push_right = jac_p_right.T @ f_push_desired_right

    # Joint regularization and torque composition
    q_target_left = qpos_left_saved
    q_target_right = qpos_right_saved
    q_current_left = get_qpos_with_names(model, data, names=joint_names_left)
    q_current_right = get_qpos_with_names(model, data, names=joint_names_right)
    q_error_left = q_target_left - q_current_left
    q_error_right = q_target_right - q_current_right
    q_error_derivative_left = -get_qvel_with_names(model, data, names=joint_names_left)
    q_error_derivative_right = -get_qvel_with_names(model, data, names=joint_names_right)
    torque_qpos_left = Kp_qpos * q_error_left + Kd_qpos * q_error_derivative_left
    torque_qpos_right = Kp_qpos * q_error_right + Kd_qpos * q_error_derivative_right
    torque_qpos_left = torque_qpos_left * 10
    torque_qpos_right = torque_qpos_right * 10

    joint_dofadr_left = [model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)] for name in joint_names_left]
    joint_dofadr_right = [model.jnt_dofadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name)] for name in joint_names_right]
    q_frc_bias_left = data.qfrc_bias[joint_dofadr_left]
    q_frc_bias_right = data.qfrc_bias[joint_dofadr_right]

    torque_ee_left = torque_ee_left * 20.0
    torque_ee_right = torque_ee_right * 20.0
    calculated_torque_left = (torque_ee_left + torque_push_left + torque_qpos_left) * 0.1
    calculated_torque_right = (torque_ee_right + torque_push_right + torque_qpos_right) * 0.1
    total_torque_left = calculated_torque_left + q_frc_bias_left
    total_torque_right = calculated_torque_right + q_frc_bias_right
    apply_ctrl_names(model, data, names=actuator_names_left, value=total_torque_left)
    apply_ctrl_names(model, data, names=actuator_names_right, value=total_torque_right)

    mujoco.mj_step(model, data)
    if render_tick % 10 == 0:
        viewer.render()
    render_tick += 1

viewer.close()
del(viewer)

2026-04-10 11:03:57.169 python[9541:339202] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit
2026-04-10 11:03:57.171 python[9541:339202] error messaging the mach port for IMKCFRunLoopWakeUpReliable
